In [ ]:
# https://www.youtube.com/watch?v=Ji3_VX80YJg

# from google.colab import drive
# drive.mount('/content/drive')


In [2]:
print('Hi')

Hi


In [3]:
import pandas as pd

df = pd.read_csv(r'D:\Data_Analysis_Science\Bert_Log_classsification_github\Bert_Model\training\dataset\synthetic_logs.csv')
df.head()

,timestamp,source,log_message,target_label
0,27-06-2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert
3,12-07-2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status
4,02-06-2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status


In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-V2")

c:\Users\veere\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7377.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-V2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:


embeddings = model.encode(df['log_message'].to_list())

In [6]:
from sklearn.cluster import DBSCAN
dbscan = DBSCAN(eps=0.5,min_samples=5,metric='euclidean')
clusters = dbscan.fit_predict(embeddings)
df['clusters'] = clusters
df['clusters'].unique()

array([ 0, -1,  1,  2,  3, 26,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,
       15, 16, 17, 18, 19, 23, 20, 21, 22, 25, 24, 27])

In [7]:
df

,timestamp,source,log_message,target_label,clusters
0,27-06-2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,-1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,-1
3,12-07-2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,0
4,02-06-2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,0
...,...,...,...,...,...
2405,13-08-2025 07:29,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,0
2406,01-11-2025 05:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,-1
2407,03-08-2025 03:07,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,9
2408,11-11-2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,-1


In [8]:
# i want to see the 5 log msgs of clusters whose no of count >10

In [9]:
cluster_counts = df['clusters'].value_counts()
large_clusters = cluster_counts[cluster_counts>10].index

for cluster in large_clusters:
  print(f'cluster {cluster}')
  print(df[df['clusters']==cluster]['log_message'].head(1))
  print()

cluster 0
0    nova.osapi_compute.wsgi.server [req-b9718cd8-f...
Name: log_message, dtype: object

cluster -1
1    Email service experiencing issues with sending
Name: log_message, dtype: object

cluster 9
31    nova.metadata.wsgi.server [-] 10.11.21.138,10....
Name: log_message, dtype: object

cluster 7
27    User User685 logged out.
Name: log_message, dtype: object

cluster 8
30    Backup started at 2025-05-14 07:06:55.
Name: log_message, dtype: object

cluster 3
8    nova.compute.claims [req-a07ac654-8e81-416d-bf...
Name: log_message, dtype: object

cluster 4
15    Backup completed successfully.
Name: log_message, dtype: object

cluster 13
50    System updated to version 3.9.1.
Name: log_message, dtype: object

cluster 2
7    File data_6169.csv uploaded successfully by us...
Name: log_message, dtype: object

cluster 16
96    Disk cleanup completed successfully.
Name: log_message, dtype: object

cluster 11
36    System reboot initiated by user User243.
Name: log_message, dtype: objec

In [10]:
import re
def classify_with_regex(log_message):
  regex_patterns = {
      r'User User\d+ logged (out|in).': 'User Action',
      r'Backup (Started|ended) at .*': 'System Notification',
      r'Backup completed successfully.': 'System Notification',
      r'Systerm updated to version .*': 'System Notification',
      r'File .* uploaded successfully by user .*': 'System Notification',
      r'Disk cleanup completed successfully.': 'System Notification',
      r'System bootup intiated by user .*': 'System Notification',
      r'Account with ID .* created by .*': 'User Action'}
  for pattern,label in regex_patterns.items():
    if re.search(pattern,log_message,re.IGNORECASE):
          return label
  #?return 'other'
    else:
      None




In [11]:
print(classify_with_regex('User User685 logged In.'))

User Action


In [12]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)
df_non_regex = df[df['regex_label'].isna()].copy()
df_non_regex

,timestamp,source,log_message,target_label,clusters,regex_label
0,27-06-2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,0,None
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,-1,None
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,-1,None
3,12-07-2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,0,None
4,02-06-2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,0,None
...,...,...,...,...,...,...
2405,13-08-2025 07:29,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,0,None
2406,01-11-2025 05:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,-1,None
2407,03-08-2025 03:07,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,9,None
2408,11-11-2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,-1,None


In [13]:
df[df['regex_label'].notnull()]

,timestamp,source,log_message,target_label,clusters,regex_label
7,10-11-2025 08:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,2,System Notification
14,01-04-2025 01:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,2,System Notification
15,05-01-2025 09:41,ModernCRM,Backup completed successfully.,System Notification,4,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,5,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,7,User Action
...,...,...,...,...,...,...
2368,12/13/2025 20:04,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,16,System Notification
2381,09-05-2025 06:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,16,System Notification
2394,04-03-2025 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,16,System Notification
2395,05-02-2025 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,8,System Notification


**Bert for unlabelled log_messages ie regex_label =None**



The log messages which less training samples i.e,cluster counts <10 will be used by LLM prompts for labelling those log msgs.
while the log msgs with enough training samples will be used by
BERT for classifying the log msgs

In [14]:
cluster_counts = df_non_regex['clusters'].value_counts()

small_clusters = cluster_counts[cluster_counts<10]

for cluster in small_clusters:
  print(f'cluster {cluster}')
  print(df_non_regex[df_non_regex['clusters']==cluster]['log_message'].head(1))
  print()

cluster 8
Series([], Name: log_message, dtype: object)

cluster 7
Series([], Name: log_message, dtype: object)

cluster 7
Series([], Name: log_message, dtype: object)

cluster 7
Series([], Name: log_message, dtype: object)

cluster 6
22    nova.compute.resource_tracker [req-addc1839-2e...
Name: log_message, dtype: object

cluster 6
22    nova.compute.resource_tracker [req-addc1839-2e...
Name: log_message, dtype: object



In [15]:
df.groupby(['source'])['clusters'].sum()

source
AnalyticsEngine    1845
BillingSystem      1850
LegacyCRM            -7
ModernCRM          1648
ModernHR           1933
ThirdPartyAPI      2255
Name: clusters, dtype: int64

from the above series it is evident that 'dpereciation_warning' and 'workflow Error' which have source = "LegacyCRM" and have the lowest samples . so we use these for LLM prompting

In [16]:
df_non_legacy = df_non_regex[df_non_regex['source'] != 'LegacyCRM']
df_non_legacy['source'].unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI'], dtype=object)

Generate embedding vectors for the df_non_legacy for training the Logistic regression and BERT model

In [17]:
filtered_embeddings = model.encode(df_non_legacy['log_message'].to_list())


In [18]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score

X = filtered_embeddings
y = df_non_legacy['target_label']

X_train,X_test,y_train ,y_test = train_test_split(X,y,test_size=0.25,random_state=42)
clf = LogisticRegression(max_iter=100)
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)
report=classification_report(y_pred,y_test)
print(report)

                     precision    recall  f1-score   support

     Critical Error       1.00      0.97      0.99        35
              Error       0.98      0.96      0.97        47
        HTTP Status       1.00      1.00      1.00       253
     Resource Usage       1.00      1.00      1.00        43
     Security Alert       0.98      1.00      0.99       106
System Notification       1.00      1.00      1.00        19

           accuracy                           0.99       503
          macro avg       0.99      0.99      0.99       503
       weighted avg       0.99      0.99      0.99       503



save the LR model into the models folder

In [21]:
import joblib
joblib.dump(clf,r'D:\Data_Analysis_Science\Bert_Log_classsification_github\Bert_Model\models\log_classifier.joblib')

['D:\\Data_Analysis_Science\\Bert_Log_classsification_github\\Bert_Model\\models\\log_classifier.joblib']

In [20]:
df.groupby('source')['log_message'].sum()

source
AnalyticsEngine    Unauthorized access to data was attemptednova....
BillingSystem      nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...
LegacyCRM          Lead conversion failed for prospect ID 7842 du...
ModernCRM          nova.osapi_compute.wsgi.server [req-b9718cd8-f...
ModernHR           nova.osapi_compute.wsgi.server [req-4895c258-b...
ThirdPartyAPI      nova.compute.claims [req-a07ac654-8e81-416d-bf...
Name: log_message, dtype: object